# 🚁 Detección de Marcadores ArUco para Localización de Drones en Almacén
## Guía Universitaria Completa — Visión por Computadora Aplicada

---

> **Asignatura:** Robótica y Sistemas Autónomos  
> **Nivel:** Grado / Máster en Ingeniería  
> **Duración estimada:** 2–3 horas  
> **Requisitos previos:** Python básico, álgebra lineal elemental  

---

### ¿Qué aprenderás en este notebook?

Al terminar esta guía serás capaz de:

1. Explicar **qué es un marcador ArUco** y por qué se usa en robótica.
2. Usar **OpenCV** para detectar marcadores en imágenes reales.
3. Entender cómo se calcula la **posición 3-D** de un marcador con la cámara.
4. Construir un **mini-mapa** del almacén y actualizar la posición del dron.
5. Analizar los **límites del sistema**: distancia, ángulo, iluminación.

---

### Índice

| # | Sección | Contenido |
|---|---------|-----------|
| 1 | Preparación del entorno | Librerías, configuración |
| 2 | ¿Qué es un marcador ArUco? | Teoría + diccionarios |
| 3 | La cámara como sensor | Modelo pinhole, matriz K |
| 4 | Detección paso a paso | De píxeles a ID |
| 5 | Estimación de pose 3-D | solvePnP, tvec, rvec |
| 6 | El mapa del almacén | Coordenadas mundo, visualización |
| 7 | Programa completo comentado | Bucle principal |
| 8 | Experimentos y límites | Distancia, ángulo, ruido |
| 9 | Ejercicios para el estudiante | TO-DO con autocorrección |


---
## Sección 1 — Preparación del Entorno

Antes de empezar, nos aseguramos de que todas las librerías necesarias están disponibles.

| Librería | Para qué la usamos |
|----------|--------------------|
| `cv2` (OpenCV) | Detección ArUco, procesado de imagen |
| `numpy` | Álgebra lineal, matrices de transformación |
| `matplotlib` | Gráficas y visualizaciones estáticas |
| `pathlib` | Manejo de rutas de archivo |

> ⚠️ Si falta alguna librería ejecuta en terminal:  
> `pip install opencv-contrib-python numpy matplotlib`


In [1]:
# ── Importaciones ──────────────────────────────────────────────────────────
import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path

# ── Configuración de matplotlib ────────────────────────────────────────────
plt.rcParams["figure.figsize"]  = (10, 5)
plt.rcParams["axes.titlesize"]  = 13
plt.rcParams["axes.labelsize"]  = 11
plt.rcParams["font.family"]     = "DejaVu Sans"

# ── Verificar versiones ────────────────────────────────────────────────────
print(f"✅ OpenCV  : {cv2.__version__}")
print(f"✅ NumPy   : {np.__version__}")
print(f"✅ Módulo ArUco disponible: {hasattr(cv2, 'aruco')}")


ModuleNotFoundError: No module named 'matplotlib'

---
## Sección 2 — ¿Qué es un Marcador ArUco?

### 2.1 Definición

Un marcador **ArUco** (*Augmented Reality University of Córdoba*) es un código binario cuadrado que una cámara puede detectar y decodificar de forma robusta, incluso con perspectiva, oclusiones parciales o mala iluminación.

```
┌─────────────────────┐
│  ░░░░░░░░░░░░░░░    │   ← borde negro (quiet zone)
│  ░ ██ █  ██ ██ ░   │
│  ░ ██ ██ █  █  ░   │   ← bits de datos (0 o 1)
│  ░ █  █  ██ ██ ░   │
│  ░░░░░░░░░░░░░░░    │
└─────────────────────┘
       Marcador 4×4
```

### 2.2 Anatomía de un marcador

| Parte | Descripción |
|-------|-------------|
| **Borde exterior negro** | Detecta que hay un marcador (contraste) |
| **Celda interior n×n** | Almacena el ID en binario |
| **Bits de corrección** | Permiten recuperar el ID aunque haya ruido |

### 2.3 Diccionarios ArUco

Un **diccionario** define cuántos marcadores distintos existen y cómo están codificados. Usamos `DICT_4X4_1000`:

- `4×4` → la cuadrícula de bits es de 4×4 = 16 bits
- `1000` → hay 1000 IDs posibles (0 al 999)
- En nuestro proyecto solo usamos los **IDs 0 al 4**

### 2.4 ¿Por qué son mejores que un simple QR?

Los ArUco están optimizados para:
- **Velocidad**: detección en tiempo real (>30 fps)
- **Robustez angular**: funcionan hasta ~60° de inclinación
- **Estimación de pose**: además del ID, dan la posición 3-D exacta


In [ ]:
# ── Generar y visualizar los 5 marcadores de nuestro proyecto ──────────────

diccionario = cv2.aruco.getPredefinedDictionary(cv2.aruco.DICT_4X4_1000)

fig, axes = plt.subplots(1, 5, figsize=(14, 3))
fig.suptitle("Nuestros 5 marcadores ArUco  (DICT_4X4_1000, IDs 0–4)",
             fontsize=14, fontweight="bold")

for marker_id in range(5):
    # Generar imagen del marcador (200x200 px)
    img = cv2.aruco.generateImageMarker(diccionario, marker_id, 200)

    axes[marker_id].imshow(img, cmap="gray", vmin=0, vmax=255)
    axes[marker_id].set_title(f"ID: {marker_id}", fontsize=13)
    axes[marker_id].axis("off")

    # Señalar la "quiet zone" (borde blanco exterior)
    rect = plt.Rectangle((0, 0), 199, 199,
                          linewidth=3, edgecolor="red", facecolor="none")
    axes[marker_id].add_patch(rect)

fig.text(0.5, -0.02,
         "El recuadro ROJO indica el borde negro que OpenCV busca primero.",
         ha="center", fontsize=10, color="red")
plt.tight_layout()
plt.show()

print("\nNota: estas imágenes son idénticas a tus archivos SVG  4x4_1000-0.svg … 4x4_1000-4.svg")


---
## Sección 3 — La Cámara como Sensor: El Modelo Pinhole

### 3.1 ¿Cómo ve la cámara el mundo?

Una cámara transforma puntos 3-D del mundo en píxeles 2-D de la imagen. El modelo más sencillo y más usado es el **modelo pinhole** (estenopeico):

$$
\begin{bmatrix} u \\ v \\ 1 \end{bmatrix}
=
\frac{1}{Z}
\underbrace{
\begin{bmatrix} f_x & 0   & c_x \\ 0   & f_y & c_y \\ 0   & 0   & 1  \end{bmatrix}
}_{\mathbf{K} \text{ — Matriz intrínseca}}
\begin{bmatrix} X \\ Y \\ Z \end{bmatrix}
$$

Donde:
- $(u, v)$ → píxel en la imagen
- $(X, Y, Z)$ → punto 3-D en el sistema de la **cámara** (metros)
- $f_x, f_y$ → distancia focal en píxeles
- $c_x, c_y$ → punto principal (centro óptico proyectado en la imagen)

### 3.2 La matriz intrínseca K

**K** caracteriza la geometría interna de la cámara. Se obtiene mediante **calibración** (proceso de la sección de calibración del proyecto).

$$
\mathbf{K} = \begin{bmatrix} 921 & 0 & 460 \\ 0 & 919 & 351 \\ 0 & 0 & 1 \end{bmatrix}
\quad \text{(ejemplo para cámara 1280×720)}
$$

> 💡 **Clave para entender la detección ArUco:**  
> Conociendo K y las 4 esquinas del marcador en la imagen (píxeles),  
> podemos deshacer la proyección y recuperar la posición 3-D del marcador.

### 3.3 Pasos del procesado ArUco

```
Frame de vídeo
     │
     ▼
[1] Escala de grises     → reducir datos innecesarios
     │
     ▼
[2] Umbralización adaptativa → detectar contornos cuadrados
     │
     ▼
[3] Filtrar cuadrados    → por tamaño, relación de aspecto
     │
     ▼
[4] Decodificar bits     → comparar con el diccionario → obtener ID
     │
     ▼
[5] Refinar esquinas     → subpíxel (mayor precisión)
     │
     ▼
[6] solvePnP             → calcular posición 3-D
```


In [ ]:
# ── Visualizar la proyección pinhole ──────────────────────────────────────
# Simulamos cómo un punto 3-D se convierte en un píxel 2-D

# Parámetros de cámara de ejemplo (típicos para 1280×720)
K = np.array([
    [921.17, 0.0,    459.90],
    [0.0,    919.02, 351.24],
    [0.0,    0.0,    1.0   ]
], dtype=np.float64)

print("Matriz intrínseca K:")
print(K)
print()

# Esquina superior-izquierda de un marcador de 15 cm a 1 metro de distancia
X, Y, Z = -0.075, 0.075, 1.0   # metros

# Proyección manual
u = K[0,0] * X / Z + K[0,2]
v = K[1,1] * Y / Z + K[1,2]

print(f"Punto 3-D en cámara : ({X}, {Y}, {Z}) m")
print(f"Proyectado en imagen: u={u:.1f} px, v={v:.1f} px")

# Visualización del plano imagen
fig, ax = plt.subplots(figsize=(8, 5))
ax.set_xlim(0, 1280); ax.set_ylim(720, 0)  # origen en esquina sup-izq
ax.set_facecolor("#1a1a2e")
ax.set_title("Plano imagen de la cámara (1280×720 px)", fontsize=13)
ax.set_xlabel("u (píxeles →)"); ax.set_ylabel("v (píxeles ↓)")

# Punto principal
ax.scatter(K[0,2], K[1,2], color="yellow", s=80, zorder=5, label=f"Punto principal ({K[0,2]:.0f},{K[1,2]:.0f})")

# Simular las 4 esquinas de un marcador a 1 m
half = 0.075
corners_3d = [(-half, half, 1.0), (half, half, 1.0),
               (half, -half, 1.0), (-half, -half, 1.0)]
corners_px = []
for (cx, cy, cz) in corners_3d:
    cu = K[0,0]*cx/cz + K[0,2]
    cv = K[1,1]*cy/cz + K[1,2]
    corners_px.append((cu, cv))

xs = [c[0] for c in corners_px] + [corners_px[0][0]]
ys = [c[1] for c in corners_px] + [corners_px[0][1]]
ax.plot(xs, ys, "g-", linewidth=2)
ax.fill(xs[:-1], ys[:-1], alpha=0.2, color="green")
ax.scatter([c[0] for c in corners_px], [c[1] for c in corners_px],
           color="lime", s=60, zorder=6, label="Esquinas del marcador (ID=X)")

ax.annotate(f"Marcador 15cm\na 1m de distancia",
            xy=(corners_px[0][0], corners_px[0][1]),
            xytext=(corners_px[0][0]-120, corners_px[0][1]-40),
            color="white", fontsize=9,
            arrowprops=dict(arrowstyle="->", color="white"))

ax.legend(facecolor="#2d2d44", labelcolor="white")
plt.tight_layout()
plt.show()


---
## Sección 4 — Detección Paso a Paso

Vamos a **desmontar el proceso de detección** ejecutando cada etapa por separado sobre una imagen sintética de un marcador.

### 4.1 Etapas internas de `detectMarkers()`

OpenCV hace todo esto en una sola llamada, pero internamente sigue estos pasos:

| Etapa | Función interna | Resultado |
|-------|----------------|-----------|
| 1. Gris | `cvtColor(BGR→GRAY)` | Imagen en niveles de gris |
| 2. Umbral adaptativo | `adaptiveThreshold` | Imagen binaria (blanco/negro) |
| 3. Contornos | `findContours` | Lista de bordes |
| 4. Filtrar cuadrados | Comprobar 4 vértices | Candidatos a marcador |
| 5. Decodificar | Comparar bits con diccionario | ID del marcador |
| 6. Refinar | `cornerSubPix` | Esquinas en subpíxel |


In [ ]:
# ── Simular cada etapa de detección sobre un marcador sintético ────────────

# PASO 0: Crear imagen de prueba (marcador ID=2 sobre fondo gris)
diccionario = cv2.aruco.getPredefinedDictionary(cv2.aruco.DICT_4X4_1000)
marker_img  = cv2.aruco.generateImageMarker(diccionario, 2, 150)

# Incrustar en un fondo gris más grande
scene = np.full((400, 600), 180, dtype=np.uint8)
scene[125:275, 225:375] = marker_img

# Añadir un poco de ruido gaussiano (simula condición real)
ruido = np.random.normal(0, 12, scene.shape).astype(np.int16)
scene = np.clip(scene.astype(np.int16) + ruido, 0, 255).astype(np.uint8)

# PASO 1: Umbralización adaptativa
umbral = cv2.adaptiveThreshold(
    scene, 255,
    cv2.ADAPTIVE_THRESH_MEAN_C,
    cv2.THRESH_BINARY_INV,
    blockSize=21, C=7
)

# PASO 2: Encontrar contornos
contornos, _ = cv2.findContours(umbral, cv2.RETR_LIST, cv2.CHAIN_APPROX_SIMPLE)

# Dibujar todos los contornos en una imagen de color
contorno_vis = cv2.cvtColor(scene, cv2.COLOR_GRAY2BGR)
cv2.drawContours(contorno_vis, contornos, -1, (0, 100, 255), 1)

# PASO 3: Detección completa con OpenCV
parametros = cv2.aruco.DetectorParameters()
detector   = cv2.aruco.ArucoDetector(diccionario, parametros)

scene_bgr  = cv2.cvtColor(scene, cv2.COLOR_GRAY2BGR)
esquinas, ids, rechazados = detector.detectMarkers(scene_bgr)

resultado = scene_bgr.copy()
if ids is not None:
    cv2.aruco.drawDetectedMarkers(resultado, esquinas, ids)
    cx = int(esquinas[0][0][:, 0].mean())
    cy = int(esquinas[0][0][:, 1].mean())
    cv2.putText(resultado, f"ID: {ids[0][0]}", (cx - 20, cy - 15),
                cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 255, 0), 2)

# ── Mostrar las 4 etapas ────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
fig.suptitle("Las 4 etapas de la detección ArUco", fontsize=14, fontweight="bold")

datos = [
    (scene,       "gray", "Paso 0: Frame original\n(gris + ruido)"),
    (umbral,      "gray", "Paso 1: Umbral adaptativo\n(imagen binaria)"),
    (cv2.cvtColor(contorno_vis, cv2.COLOR_BGR2RGB), None,
                         "Paso 2: Contornos detectados\n(en naranja)"),
    (cv2.cvtColor(resultado, cv2.COLOR_BGR2RGB),    None,
                         "Paso 3: Marcador identificado\n(ID=2 ✅)"),
]

for ax, (img, cmap, titulo) in zip(axes, datos):
    ax.imshow(img, cmap=cmap)
    ax.set_title(titulo, fontsize=10)
    ax.axis("off")

plt.tight_layout()
plt.show()


---
## Sección 5 — Estimación de Pose 3-D con solvePnP

### 5.1 El problema PnP (Perspective-n-Point)

Una vez conocemos las **4 esquinas del marcador en píxeles** y conocemos su **tamaño físico real** (15 cm), podemos resolver el sistema:

$$
\underbrace{s \begin{bmatrix}u_i \\ v_i \\ 1\end{bmatrix}}_{\text{píxeles}} 
= \mathbf{K} \underbrace{\begin{bmatrix}\mathbf{R} \mid \mathbf{t}\end{bmatrix}}_{\text{pose}} 
\underbrace{\begin{bmatrix}X_i \\ Y_i \\ Z_i \\ 1\end{bmatrix}}_{\text{puntos 3-D del marcador}}
$$

Con 4 puntos tenemos **8 ecuaciones** para 6 incógnitas (3 rotación + 3 traslación). El sistema está sobredeterminado → solución por mínimos cuadrados.

### 5.2 ¿Qué devuelve solvePnP?

| Variable | Tipo | Significado |
|----------|------|-------------|
| `rvec` | array (3,1) | Vector de **rotación** (Rodrigues) — orienta la cámara |
| `tvec` | array (3,1) | Vector de **traslación** — posición de la cámara respecto al marcador (metros) |

### 5.3 Interpretar tvec

$$
\mathbf{t} = \begin{bmatrix}t_x \\ t_y \\ t_z\end{bmatrix} \approx \text{posición de la cámara respecto al marcador}
$$

- $t_z$ ≈ **distancia frontal** (cuánto está delante)
- $t_x$ ≈ desplazamiento lateral
- $t_y$ ≈ desplazamiento vertical

$$\text{distancia} = \|\mathbf{t}\| = \sqrt{t_x^2 + t_y^2 + t_z^2}$$

### 5.4 De rvec a ángulos Euler

`rvec` es un vector de Rodrigues. Para obtener ángulos legibles:
1. `cv2.Rodrigues(rvec)` → Matriz de rotación R (3×3)
2. Descomponer R en ángulos roll, pitch, yaw


In [ ]:
# ── Demostración de solvePnP sobre el marcador sintético ───────────────────

# Puntos 3-D reales de las esquinas de un marcador de 15 cm
# Orden: TL, TR, BR, BL  (igual que ArUco)
MARKER_SIZE = 0.15   # metros
half = MARKER_SIZE / 2.0
obj_points = np.array([
    [-half,  half, 0.0],   # TL  (arriba izquierda)
    [ half,  half, 0.0],   # TR  (arriba derecha)
    [ half, -half, 0.0],   # BR  (abajo derecha)
    [-half, -half, 0.0],   # BL  (abajo izquierda)
], dtype=np.float32)

# Esquinas detectadas en la imagen (en píxeles)
# Si no se detectaron en la celda anterior, usamos valores de ejemplo
if ids is not None and len(esquinas) > 0:
    img_points = esquinas[0].reshape(4, 2).astype(np.float32)
    print("✅ Usando esquinas detectadas reales")
else:
    # Ejemplo simulado: marcador a ~1m frente a la cámara
    img_points = np.array([
        [390.0, 191.0],
        [460.0, 191.0],
        [460.0, 259.0],
        [390.0, 259.0],
    ], dtype=np.float32)
    print("⚠️  Usando esquinas de ejemplo")

# ── Llamar a solvePnP ──────────────────────────────────────────────────────
dist_coeffs = np.zeros((5, 1), dtype=np.float64)  # sin distorsión (simplificado)

ok, rvec, tvec = cv2.solvePnP(
    obj_points,
    img_points,
    K,
    dist_coeffs,
    flags=cv2.SOLVEPNP_IPPE_SQUARE,
)

# ── Mostrar resultados ─────────────────────────────────────────────────────
print(f"\n{'─'*50}")
print(f"  solvePnP resuelto: {ok}")
print(f"  rvec (rotación Rodrigues): {rvec.flatten().round(3)}")
print(f"  tvec (traslación en metros):")
print(f"     tx = {tvec[0,0]:.4f} m  (lateral)")
print(f"     ty = {tvec[1,0]:.4f} m  (vertical)")
print(f"     tz = {tvec[2,0]:.4f} m  (profundidad / distancia frontal)")
distancia = float(np.linalg.norm(tvec))
print(f"  Distancia total al marcador: {distancia:.4f} m")

# ── Convertir rvec a ángulos Euler ─────────────────────────────────────────
R, _ = cv2.Rodrigues(rvec)
sy   = np.sqrt(R[0, 0]**2 + R[1, 0]**2)
roll  = np.degrees(np.arctan2( R[2, 1], R[2, 2]))
pitch = np.degrees(np.arctan2(-R[2, 0], sy))
yaw   = np.degrees(np.arctan2( R[1, 0], R[0, 0]))

print(f"\n  Orientación de la cámara (Euler ZYX):")
print(f"     Roll  (giro eje Z) = {roll:.1f}°")
print(f"     Pitch (giro eje X) = {pitch:.1f}°")
print(f"     Yaw   (giro eje Y) = {yaw:.1f}°")
print(f"{'─'*50}")


In [ ]:
# ── Visualizar los ejes de coordenadas sobre el marcador ──────────────────
# cv2.drawFrameAxes dibuja X (rojo), Y (verde), Z (azul) saliendo del marcador

vis_pose = cv2.cvtColor(scene, cv2.COLOR_GRAY2BGR)

if ids is not None:
    cv2.aruco.drawDetectedMarkers(vis_pose, esquinas, ids)
    cv2.drawFrameAxes(vis_pose, K, dist_coeffs, rvec, tvec, MARKER_SIZE * 0.6)

fig, ax = plt.subplots(figsize=(8, 5))
ax.imshow(cv2.cvtColor(vis_pose, cv2.COLOR_BGR2RGB))
ax.set_title("Ejes de coordenadas proyectados sobre el marcador\n"
             "X→ Rojo   Y→ Verde   Z→ Azul (sale hacia la cámara)", fontsize=11)
ax.axis("off")

# Leyenda manual
for color, etiqueta in [("red","X (lateral)"), ("lime","Y (vertical)"), ("blue","Z (profundidad)")]:
    ax.plot([], [], color=color, linewidth=3, label=etiqueta)
ax.legend(loc="upper right", fontsize=10)
plt.tight_layout()
plt.show()

print("💡 El eje Z (azul) apunta desde el marcador hacia la cámara.")
print("   La longitud del eje en la imagen cambia con la distancia.")


---
## Sección 6 — El Mapa del Almacén

### 6.1 Sistema de coordenadas del mundo

Definimos un origen fijo en la **esquina inferior-izquierda** del almacén:

```
Y (metros, Norte)
│
│   ID=3 ●───────────● ID=1
│        │           │
│   ID=4 ●     ●     │
│        │    ID=0   │
│   ID=2 ●───────────●  ── X (metros, Este)
(0,0)
```

Cada marcador tiene una **posición conocida** en este sistema. Cuando el dron ve un marcador, puede calcular **dónde está él** respecto al marcador, y como sabe dónde está el marcador en el mapa... ¡sabe dónde está él en el mapa!

### 6.2 Posiciones de nuestros 5 marcadores

| ID | X (m) | Y (m) | Ubicación |
|----|-------|-------|-----------|
| 0  | 0.0   | 0.0   | Esquina izquierda-abajo |
| 1  | 5.0   | 0.0   | Esquina derecha-abajo |
| 2  | 5.0   | 4.0   | Esquina derecha-arriba |
| 3  | 0.0   | 4.0   | Esquina izquierda-arriba |
| 4  | 2.5   | 2.0   | Centro |

Almacén de **5 × 4 metros**.


In [ ]:
# ── Dibujar el mapa del almacén con matplotlib ─────────────────────────────

POSICIONES_MARCADORES = {
    0: (0.0, 0.0),
    1: (5.0, 0.0),
    2: (5.0, 4.0),
    3: (0.0, 4.0),
    4: (2.5, 2.0),
}
ANCHO_MAPA, ALTO_MAPA = 5.0, 4.0

def dibujar_mapa_matplotlib(ids_detectados=None, titulo="Mapa del Almacén"):
    """
    Dibuja el mapa del almacén.
    ids_detectados: lista de IDs actualmente visibles (se colorean en verde)
    """
    if ids_detectados is None:
        ids_detectados = []

    fig, ax = plt.subplots(figsize=(8, 6.5))
    ax.set_facecolor("#1c1c2e")
    fig.patch.set_facecolor("#13131f")

    # Paredes del almacén
    rect = mpatches.FancyBboxPatch(
        (0, 0), ANCHO_MAPA, ALTO_MAPA,
        boxstyle="square,pad=0",
        edgecolor="#aaaaaa", facecolor="none", linewidth=2.5
    )
    ax.add_patch(rect)

    # Marcadores
    for mid, (mx, my) in POSICIONES_MARCADORES.items():
        detectado = mid in ids_detectados
        color   = "#00dc50" if detectado else "#555555"
        tamanio = 250 if detectado else 120
        borde   = "white" if detectado else "gray"

        ax.scatter(mx, my, s=tamanio, c=color,
                   edgecolors=borde, linewidths=1.5, zorder=5)
        ax.text(mx + 0.15, my + 0.12, f"ID {mid}",
                color="white" if detectado else "#888888",
                fontsize=11, fontweight="bold" if detectado else "normal")

        if detectado:
            # Aro de "señal activa"
            circle = plt.Circle((mx, my), 0.22, color="#00dc50",
                                 fill=False, linewidth=1.5, linestyle="--")
            ax.add_patch(circle)

    # Leyenda
    ax.scatter([], [], c="#00dc50", s=150, edgecolors="white",
               linewidths=1.5, label="Marcador detectado ✅")
    ax.scatter([], [], c="#555555", s=80, edgecolors="gray",
               linewidths=1.5, label="Marcador no visto")

    ax.set_xlim(-0.5, ANCHO_MAPA + 0.5)
    ax.set_ylim(-0.5, ALTO_MAPA + 0.5)
    ax.set_xlabel("X (metros)", color="white", fontsize=11)
    ax.set_ylabel("Y (metros)", color="white", fontsize=11)
    ax.tick_params(colors="white")
    ax.set_title(titulo, color="white", fontsize=13, fontweight="bold")
    ax.legend(facecolor="#2a2a3e", labelcolor="white", fontsize=10)
    ax.grid(True, color="#2a2a3e", linewidth=0.8)

    plt.tight_layout()
    plt.show()

# ── Mostrar el mapa vacío y luego con 2 marcadores detectados ──────────────
print("Estado 1: ningún marcador visible")
dibujar_mapa_matplotlib(ids_detectados=[], titulo="Mapa del Almacén — Sin marcadores")

print("Estado 2: se ven los marcadores 0 y 4")
dibujar_mapa_matplotlib(ids_detectados=[0, 4], titulo="Mapa del Almacén — IDs 0 y 4 detectados")


---
## Sección 7 — El Programa Completo Explicado Línea a Línea

Ahora que entendemos cada pieza, leeremos el código de `detector_simple.py` bloque a bloque y ejecutaremos una versión adaptada para el notebook.

### 7.1 Flujo del programa

```
┌─────────────────────────────────────────────────────┐
│                   BUCLE PRINCIPAL                    │
│                                                      │
│  cap.read()  ──→  frame BGR                          │
│       │                                              │
│       ▼                                              │
│  detector.detectMarkers(frame)                       │
│       │                                              │
│       ├── esquinas: list[ array(1,4,2) ]             │
│       ├── ids:      array(N,1) con los IDs           │
│       └── rechazados: candidatos descartados         │
│                                                      │
│  si ids != None:                                     │
│       ▼                                              │
│  drawDetectedMarkers()  → dibuja recuadros           │
│  Para cada id:                                       │
│       putText(ID)       → etiqueta en pantalla       │
│                                                      │
│  dibujar_mapa(ids_lista) → panel derecho             │
│                                                      │
│  imshow(cámara + mapa)  → mostrar ventana            │
└─────────────────────────────────────────────────────┘
```


In [ ]:
# ── Versión del programa adaptada para notebook ────────────────────────────
# (procesa imágenes estáticas en lugar de vídeo en directo)

def procesar_frame(frame_bgr, detector, verbose=True):
    """
    Función equivalente a una iteración del bucle principal.
    Devuelve: frame anotado, lista de IDs detectados.
    """
    # ── PASO 1: Detectar marcadores ──────────────────────────────────────
    esquinas, ids, rechazados = detector.detectMarkers(frame_bgr)
    #   esquinas  → lista de N arrays, cada uno (1, 4, 2)
    #   ids       → array (N, 1) o None
    #   rechazados→ candidatos que no pasaron la decodificación

    ids_lista = []

    # ── PASO 2: Anotar el frame ──────────────────────────────────────────
    frame_anot = frame_bgr.copy()

    if ids is not None:
        # Dibujar recuadros y esquinas
        cv2.aruco.drawDetectedMarkers(frame_anot, esquinas, ids)

        ids_lista = ids.flatten().tolist()

        for i, marker_id in enumerate(ids_lista):
            # Centro del marcador
            cx = int(esquinas[i][0][:, 0].mean())
            cy = int(esquinas[i][0][:, 1].mean())

            # Etiqueta con ID
            cv2.putText(frame_anot, f"ID: {int(marker_id)}",
                        (cx - 20, cy - 15),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)

    # Texto de estado
    estado = f"Detectados: {[int(x) for x in ids_lista]}" if ids_lista \
             else "Buscando marcadores..."
    cv2.putText(frame_anot, estado, (10, 30),
                cv2.FONT_HERSHEY_SIMPLEX, 0.65, (0, 200, 255), 2)

    if verbose:
        print(f"  IDs detectados : {[int(x) for x in ids_lista]}")
        print(f"  Rechazados     : {len(rechazados)} candidatos descartados")

    return frame_anot, ids_lista


# ── Crear imagen de prueba con VARIOS marcadores ───────────────────────────
def crear_escena_multi(ids_a_poner, tamanio_marker=120, ruido=8):
    """Genera una imagen con varios marcadores colocados en posiciones fijas."""
    diccionario = cv2.aruco.getPredefinedDictionary(cv2.aruco.DICT_4X4_1000)
    scene = np.full((480, 640, 3), (50, 50, 50), dtype=np.uint8)

    posiciones = [(30, 30), (320, 30), (30, 260), (320, 260), (175, 145)]
    for i, mid in enumerate(ids_a_poner[:5]):
        x0, y0 = posiciones[i]
        img = cv2.aruco.generateImageMarker(diccionario, mid, tamanio_marker)
        img_bgr = cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)
        h, w    = img_bgr.shape[:2]
        x1, y1  = min(x0 + w, 640), min(y0 + h, 480)
        scene[y0:y1, x0:x1] = img_bgr[:y1-y0, :x1-x0]

    # Ruido para simular condiciones reales
    noise = np.random.normal(0, ruido, scene.shape).astype(np.int16)
    scene = np.clip(scene.astype(np.int16) + noise, 0, 255).astype(np.uint8)
    return scene

# ── Ejecutar sobre la escena sintética ────────────────────────────────────
parametros_det = cv2.aruco.DetectorParameters()
detector_nb    = cv2.aruco.ArucoDetector(diccionario, parametros_det)

print("=== Procesando escena con marcadores 0, 2, 4 ===")
escena   = crear_escena_multi([0, 2, 4])
resultado, ids_encontrados = procesar_frame(escena, detector_nb)

# Mostrar resultado
fig, ax = plt.subplots(figsize=(9, 6))
ax.imshow(cv2.cvtColor(resultado, cv2.COLOR_BGR2RGB))
ax.set_title(f"Frame procesado — IDs encontrados: {ids_encontrados}", fontsize=12)
ax.axis("off")
plt.tight_layout()
plt.show()


---
## Sección 8 — Experimentos: ¿Cuáles son los Límites del Sistema?

### 8.1 ¿Qué factores afectan a la detección?

| Factor | Efecto | Valor crítico |
|--------|--------|---------------|
| **Distancia** | A mayor distancia, el marcador ocupa menos píxeles | >8m con marcador 15cm |
| **Ángulo** | La perspectiva deforma los bits → errores de decodificación | >70° |
| **Ruido / iluminación** | El umbral adaptativo puede fallar | <50 lux |
| **Desenfoque** | Reduce la nitidez de los bordes | Velocidad >2 m/s sin obturador rápido |
| **Tamaño del marcador** | Marcador más grande → mayor rango de detección | Proporcional a √(pixels) |

### 8.2 Regla práctica de distancia máxima

Empíricamente, para que un marcador sea detectable necesita aparecer con al menos **20×20 píxeles** en la imagen.

$$d_{max} \approx \frac{f_x \cdot L_{marcador}}{20 \text{ px}}$$

Para nuestra cámara (fx ≈ 921) y marcador de 15 cm:

$$d_{max} \approx \frac{921 \times 0.15}{20} \approx 6.9 \text{ m}$$


In [ ]:
# ── Experimento 1: efecto del RUIDO en la detección ───────────────────────

niveles_ruido = [0, 10, 25, 45, 70, 100]
tasas          = []

print("Experimento: ¿a partir de qué nivel de ruido falla la detección?")
print(f"{'Ruido (σ)':>12} │ {'Detecciones':>11} │ {'Estado':>8}")
print("─" * 40)

for sigma in niveles_ruido:
    detectados_en_prueba = 0
    N_PRUEBAS = 20
    for _ in range(N_PRUEBAS):
        escena = crear_escena_multi([0, 1, 2], ruido=sigma)
        _, ids_enc = procesar_frame(escena, detector_nb, verbose=False)
        detectados_en_prueba += len(ids_enc)

    tasa = detectados_en_prueba / (N_PRUEBAS * 3)  # 3 marcadores esperados
    tasas.append(tasa)
    estado = "✅ OK" if tasa > 0.8 else ("⚠️  Degradado" if tasa > 0.3 else "❌ Fallo")
    print(f"{sigma:>12} │ {tasa*100:>10.0f}% │ {estado}")

# Gráfica
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(niveles_ruido, [t * 100 for t in tasas], "o-",
        color="#00dc50", linewidth=2.5, markersize=8)
ax.axhline(80, color="orange", linestyle="--", label="Umbral aceptable (80%)")
ax.axhline(30, color="red",    linestyle="--", label="Umbral crítico (30%)")
ax.fill_between(niveles_ruido, [t*100 for t in tasas], 0,
                alpha=0.15, color="#00dc50")
ax.set_xlabel("Nivel de ruido gaussiano (σ en niveles de gris)", fontsize=11)
ax.set_ylabel("Tasa de detección (%)", fontsize=11)
ax.set_title("Robustez de la detección ArUco ante ruido de imagen", fontsize=12)
ax.set_ylim(0, 110)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
# ── Experimento 2: efecto del TAMAÑO del marcador (simula distancia) ───────

tamanios_px = [10, 15, 20, 30, 50, 80, 120, 180]
tasas_tam    = []

print("Experimento: ¿cuántos píxeles necesita el marcador para ser detectado?")
print(f"{'Tamaño (px)':>12} │ {'Detectado':>10}")
print("─" * 28)

for tam in tamanios_px:
    detectado = False
    for _ in range(5):
        dic = cv2.aruco.getPredefinedDictionary(cv2.aruco.DICT_4X4_1000)
        img_m = cv2.aruco.generateImageMarker(dic, 0, tam)
        escena_mini = np.full((300, 300, 3), 180, dtype=np.uint8)
        y0 = (300 - tam) // 2
        x0 = (300 - tam) // 2
        escena_mini[y0:y0+tam, x0:x0+tam] = cv2.cvtColor(img_m, cv2.COLOR_GRAY2BGR)
        _, ids_t = procesar_frame(escena_mini, detector_nb, verbose=False)
        if ids_t:
            detectado = True
            break

    tasas_tam.append(1 if detectado else 0)
    print(f"{tam:>12} │ {'✅ Sí' if detectado else '❌ No':>10}")

# Gráfica visual
fig, ax = plt.subplots(figsize=(9, 4))
colores = ["#00dc50" if t else "#ff4444" for t in tasas_tam]
bars    = ax.bar([str(t) for t in tamanios_px], tasas_tam, color=colores, edgecolor="white")

# Etiquetas
for bar, t, tam in zip(bars, tasas_tam, tamanios_px):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
            "✅" if t else "❌", ha="center", fontsize=14)

ax.set_xlabel("Tamaño del marcador en imagen (píxeles)", fontsize=11)
ax.set_ylabel("Detectado (1=Sí, 0=No)", fontsize=11)
ax.set_title("Tamaño mínimo de marcador para detección fiable\n"
             "(equivale a la distancia máxima de operación)", fontsize=12)
ax.set_ylim(0, 1.3)
ax.grid(True, axis="y", alpha=0.3)

# Calcular distancia equivalente
fx = 921.17; L = 0.15
print(f"\nDistancias equivalentes (fx={fx:.0f}, L={L*100:.0f}cm):")
for tam in tamanios_px:
    d = fx * L / tam
    print(f"  {tam:>4} px → d ≈ {d:.1f} m")

plt.tight_layout()
plt.show()


---
## Sección 9 — Ejercicios para el Estudiante 🎓

Cada ejercicio tiene:
- Un enunciado claro
- Código **parcialmente completado** con bloques `# TODO`
- Una celda de **autocorrección** al final

> ✏️ **Instrucción:** Completa los bloques marcados con `# TODO` y ejecuta la celda de verificación.

---

### Ejercicio 1 — Reconocer el diccionario correcto

**Enunciado:** Crea el detector ArUco apropiado para los marcadores del proyecto (DICT_4X4_1000) y comprueba que devuelve el ID correcto para una imagen generada.


In [ ]:
# ═══════════════════════════════════════════════════════════
# EJERCICIO 1 — Completa el código
# ═══════════════════════════════════════════════════════════

# Queremos detectar el marcador con ID = 3

# TODO 1: Elige el diccionario correcto (pista: los SVG son 4x4_1000-X.svg)
mi_diccionario = cv2.aruco.getPredefinedDictionary(
    cv2.aruco.DICT_4X4_1000   # ← cambia esto si crees que hay otro diccionario
)

# TODO 2: Genera una imagen del marcador con ID = 3 de 200×200 píxeles
imagen_marcador = cv2.aruco.generateImageMarker(mi_diccionario, 3, 200)

# TODO 3: Convierte la imagen a BGR (OpenCV trabaja en BGR por defecto)
imagen_bgr = cv2.cvtColor(imagen_marcador, cv2.COLOR_GRAY2BGR)

# (El detector ya está creado, no lo toques)
params  = cv2.aruco.DetectorParameters()
det_ej1 = cv2.aruco.ArucoDetector(mi_diccionario, params)

# Detectar
esq, ids_ej1, _ = det_ej1.detectMarkers(imagen_bgr)

# Mostrar resultado
fig, ax = plt.subplots(figsize=(4, 4))
ax.imshow(imagen_marcador, cmap="gray")
ax.set_title("Marcador generado (¿lo detecta?)", fontsize=11)
ax.axis("off")
plt.tight_layout()
plt.show()

# ─── NO MODIFICAR: celda de verificación ──────────────────
print("\n🔍 VERIFICACIÓN EJERCICIO 1:")
if ids_ej1 is not None and 3 in ids_ej1.flatten():
    print("  ✅ ¡Correcto! El detector identificó el marcador ID=3")
else:
    print("  ❌ No se detectó el marcador ID=3. Revisa el diccionario o el ID generado.")
    print(f"     IDs detectados: {ids_ej1}")


### Ejercicio 2 — Calcular el centro de un marcador

**Enunciado:** Dado un array de esquinas de un marcador, calcula su **centro en píxeles** usando las 4 esquinas. Luego dibuja un punto rojo en ese centro.

Pista: El centro es la media aritmética de las coordenadas X e Y de las 4 esquinas.

$$cx = \frac{x_0 + x_1 + x_2 + x_3}{4}, \quad cy = \frac{y_0 + y_1 + y_2 + y_3}{4}$$


In [ ]:
# ═══════════════════════════════════════════════════════════
# EJERCICIO 2 — Completa el código
# ═══════════════════════════════════════════════════════════

# Datos de partida: esquinas de un marcador detectado (en píxeles)
# Cada fila es una esquina: [x, y]
esquinas_ejemplo = np.array([
    [100.0, 80.0],    # esquina 0 (arriba-izquierda)
    [200.0, 80.0],    # esquina 1 (arriba-derecha)
    [200.0, 180.0],   # esquina 2 (abajo-derecha)
    [100.0, 180.0],   # esquina 3 (abajo-izquierda)
])

# TODO: Calcula el centro del marcador
# Pista: usa np.mean() sobre el eje correcto
cx = int(np.mean(esquinas_ejemplo[:, 0]))   # ← usa la columna 0 (coordenada X)
cy = int(np.mean(esquinas_ejemplo[:, 1]))   # ← usa la columna 1 (coordenada Y)

# TODO: Crea la imagen y dibuja el marcador + el punto central en ROJO
img_ej2 = np.full((300, 300, 3), 60, dtype=np.uint8)

# Dibujar el polígono del marcador
pts = esquinas_ejemplo.astype(np.int32).reshape((-1, 1, 2))
cv2.polylines(img_ej2, [pts], isClosed=True, color=(0, 255, 0), thickness=2)

# TODO: Dibuja el centro como un círculo rojo de radio 6
cv2.circle(img_ej2, (cx, cy), 6, (0, 0, 255), -1)    # ← (0,0,255) es ROJO en BGR

# Etiqueta
cv2.putText(img_ej2, f"Centro: ({cx},{cy})", (cx + 10, cy - 10),
            cv2.FONT_HERSHEY_SIMPLEX, 0.55, (0, 0, 255), 1)

# Mostrar
fig, ax = plt.subplots(figsize=(5, 5))
ax.imshow(cv2.cvtColor(img_ej2, cv2.COLOR_BGR2RGB))
ax.set_title("Centro del marcador (punto rojo)", fontsize=11)
ax.axis("off")
plt.tight_layout()
plt.show()

# ─── NO MODIFICAR: celda de verificación ──────────────────
print("\n🔍 VERIFICACIÓN EJERCICIO 2:")
cx_correcto = int(np.mean(esquinas_ejemplo[:, 0]))
cy_correcto = int(np.mean(esquinas_ejemplo[:, 1]))
if cx == cx_correcto and cy == cy_correcto:
    print(f"  ✅ ¡Correcto! Centro calculado: ({cx}, {cy})")
else:
    print(f"  ❌ El centro debería ser ({cx_correcto}, {cy_correcto}), obtuviste ({cx}, {cy})")


### Ejercicio 3 — Calcular la distancia al marcador

**Enunciado:** Usando el `tvec` devuelto por `solvePnP`, calcula la distancia euclídea en metros desde la cámara al marcador.

$$d = \|\mathbf{t}\| = \sqrt{t_x^2 + t_y^2 + t_z^2}$$

Luego clasifica la distancia como `"cerca"` (<1.5m), `"media"` (1.5–3m) o `"lejos"` (>3m).


In [ ]:
# ═══════════════════════════════════════════════════════════
# EJERCICIO 3 — Completa el código
# ═══════════════════════════════════════════════════════════

# Vectores tvec de ejemplo (como si vinieran de solvePnP)
casos = [
    np.array([[ 0.30], [-0.10], [ 1.20]]),   # caso A
    np.array([[ 0.50], [ 0.20], [ 2.50]]),   # caso B
    np.array([[-0.10], [ 0.05], [ 4.80]]),   # caso C
]

def calcular_distancia_y_clasificar(tvec_array):
    """
    Calcula la distancia euclídea desde la cámara al marcador
    y devuelve (distancia_metros, clasificacion).
    """
    # TODO: Calcula la norma del vector tvec
    distancia = float(np.linalg.norm(tvec_array))  # ← np.linalg.norm calcula √(tx²+ty²+tz²)

    # TODO: Clasifica la distancia
    if distancia < 1.5:
        clasificacion = "cerca"
    elif distancia <= 3.0:
        clasificacion = "media"
    else:
        clasificacion = "lejos"

    return distancia, clasificacion


# ── Probar los casos ────────────────────────────────────────────────────────
print("🔍 Resultados ejercicio 3:\n")
resultados_ej3 = []
for i, tvec_caso in enumerate(casos):
    d, clase = calcular_distancia_y_clasificar(tvec_caso)
    resultados_ej3.append((d, clase))
    icono = {"cerca": "🟢", "media": "🟡", "lejos": "🔴"}[clase]
    print(f"  Caso {chr(65+i)}: tvec = {tvec_caso.flatten().round(2)}")
    print(f"           distancia = {d:.3f} m  →  {icono} {clase}\n")

# ── Visualización ──────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 4))
letras   = [chr(65+i) for i in range(len(casos))]
colores  = ["#00dc50" if c=="cerca" else "#ffbb00" if c=="media" else "#ff4444"
            for _, c in resultados_ej3]
distancias = [d for d, _ in resultados_ej3]

bars = ax.bar(letras, distancias, color=colores, edgecolor="white", width=0.5)
ax.axhline(1.5, color="#00dc50", linestyle="--", linewidth=1.5, label="Límite cerca/media (1.5m)")
ax.axhline(3.0, color="#ffbb00", linestyle="--", linewidth=1.5, label="Límite media/lejos (3.0m)")
for bar, d in zip(bars, distancias):
    ax.text(bar.get_x() + bar.get_width()/2, d + 0.05, f"{d:.2f}m",
            ha="center", va="bottom", color="white", fontsize=11)
ax.set_ylabel("Distancia (metros)", fontsize=11)
ax.set_title("Clasificación de distancia al marcador", fontsize=12)
ax.legend(fontsize=9)
ax.set_ylim(0, max(distancias) + 0.7)
ax.grid(True, axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

# ─── NO MODIFICAR: celda de verificación ──────────────────
print("\n🔍 VERIFICACIÓN EJERCICIO 3:")
esperados = [("cerca", 1.5), ("media", 3.0), ("lejos", 3.0)]
ok_todo = True
for i, (tvec_caso, (clase_esp, _)) in enumerate(zip(casos, esperados)):
    d_real = float(np.linalg.norm(tvec_caso))
    d_calc, clase_calc = calcular_distancia_y_clasificar(tvec_caso)
    if abs(d_calc - d_real) < 0.001:
        print(f"  ✅ Caso {chr(65+i)}: distancia correcta ({d_calc:.3f} m, {clase_calc})")
    else:
        print(f"  ❌ Caso {chr(65+i)}: distancia incorrecta. Esperada {d_real:.3f} m")
        ok_todo = False
if ok_todo:
    print("\n  🎉 ¡Todos los casos correctos!")


### Ejercicio 4 — Completar la función `dibujar_mapa`

**Enunciado:** La función `dibujar_mapa_incompleta` tiene varios bloques `# TODO`. Complétalos para que:
1. Pinte los marcadores **detectados** en verde y los **no detectados** en gris.
2. Muestre el **número de marcadores visibles** en el título.
3. Añada el texto `"¡LOCALIZADO!"` si hay **≥ 2 marcadores** visibles.


In [ ]:
# ═══════════════════════════════════════════════════════════
# EJERCICIO 4 — Completa la función
# ═══════════════════════════════════════════════════════════

MAP_W_ej, MAP_H_ej = 400, 320
PAD_ej = 30

def metros_a_px_ej(x_m, y_m):
    px = int(PAD_ej + x_m / ANCHO_MAPA * (MAP_W_ej - 2 * PAD_ej))
    py = int(PAD_ej + y_m / ALTO_MAPA  * (MAP_H_ej - 2 * PAD_ej))
    return px, py

def dibujar_mapa_incompleta(ids_detectados):
    """Función de mapa con bloques TODO para completar."""
    mapa = np.full((MAP_H_ej, MAP_W_ej, 3), (30, 30, 30), dtype=np.uint8)

    # Borde del almacén
    cv2.rectangle(mapa, (PAD_ej, PAD_ej),
                  (MAP_W_ej - PAD_ej, MAP_H_ej - PAD_ej), (150, 150, 150), 2)

    for marker_id, (x_m, y_m) in POSICIONES_MARCADORES.items():
        px, py = metros_a_px_ej(x_m, y_m)

        # TODO 1: si el marcador está en ids_detectados → color verde (0,220,80) y radio 12
        #         si NO está detectado               → color gris  (80,80,80)  y radio 8
        if marker_id in ids_detectados:
            color  = (0, 220, 80)
            radius = 12
        else:
            color  = (80, 80, 80)
            radius = 8

        cv2.circle(mapa, (px, py), radius, color, -1)
        cv2.putText(mapa, str(marker_id), (px - 5, py + 5),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)

    # TODO 2: Escribe en el título cuántos marcadores se ven
    n_visibles = len(ids_detectados)
    titulo = f"Mapa  |  Visibles: {n_visibles}/5"
    cv2.putText(mapa, titulo, (PAD_ej, PAD_ej - 8),
                cv2.FONT_HERSHEY_SIMPLEX, 0.45, (200, 200, 200), 1)

    # TODO 3: Si hay 2 o más marcadores, escribe "¡LOCALIZADO!" en verde abajo
    if n_visibles >= 2:
        cv2.putText(mapa, "¡LOCALIZADO!", (MAP_W_ej // 2 - 65, MAP_H_ej - 8),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 220, 80), 2)

    return mapa


# ── Probar con diferentes casos ────────────────────────────────────────────
casos_ej4 = [
    ([], "Sin marcadores"),
    ([4], "Solo ID 4"),
    ([0, 2], "IDs 0 y 2"),
    ([0, 1, 3, 4], "IDs 0,1,3,4"),
]

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, (ids_caso, desc) in zip(axes, casos_ej4):
    mapa = dibujar_mapa_incompleta(ids_caso)
    ax.imshow(cv2.cvtColor(mapa, cv2.COLOR_BGR2RGB))
    ax.set_title(desc, fontsize=10)
    ax.axis("off")

fig.suptitle("Ejercicio 4: función dibujar_mapa completada", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

# ─── NO MODIFICAR: celda de verificación ──────────────────
print("\n🔍 VERIFICACIÓN EJERCICIO 4:")
mapa_test_1 = dibujar_mapa_incompleta([0, 1, 3])
mapa_test_0 = dibujar_mapa_incompleta([])

# Comprobar que hay verde cuando hay marcadores (píxel en posición marcador 0)
px0, py0 = metros_a_px_ej(0.0, 0.0)
pixel_detectado    = mapa_test_1[py0, px0]
pixel_no_detectado = mapa_test_0[py0, px0]

verde_ok = pixel_detectado[1] > 150   # canal verde alto
gris_ok  = pixel_no_detectado[1] < 120 and pixel_no_detectado[0] < 120

print(f"  ✅ Color verde cuando detectado: {verde_ok}")
print(f"  ✅ Color gris cuando no detectado: {gris_ok}")

if verde_ok and gris_ok:
    print("\n  🎉 ¡Ejercicio 4 completado correctamente!")
else:
    print("\n  ❌ Revisa los colores asignados a marcadores detectados/no detectados")


### Ejercicio 5 — Reto Final: Integrar todo el pipeline

**Enunciado:** Completa la función `pipeline_completo()` que recibe una lista de IDs detectados (simulando la cámara), y:
1. Busca cada ID en el diccionario de posiciones.
2. Calcula el **centroide** (media de las posiciones de todos los marcadores visibles).
3. Dibuja el mapa con el dron posicionado en el centroide.

Este centroide es una **estimación simple** de dónde está el dron (sin solvePnP).

> 🧠 **Concepto:** Si estamos frente a 2 marcadores a la misma distancia, el dron está aproximadamente en el punto medio entre ellos.


In [ ]:
# ═══════════════════════════════════════════════════════════
# EJERCICIO 5 — RETO FINAL
# ═══════════════════════════════════════════════════════════

def pipeline_completo(ids_detectados):
    """
    Estima la posición del dron como centroide de los marcadores visibles
    y devuelve el mapa OpenCV con el dron pintado.
    """
    mapa = np.full((MAP_H_ej, MAP_W_ej, 3), (30, 30, 30), dtype=np.uint8)
    cv2.rectangle(mapa, (PAD_ej, PAD_ej),
                  (MAP_W_ej - PAD_ej, MAP_H_ej - PAD_ej), (150, 150, 150), 2)

    # ── Dibujar todos los marcadores ──────────────────────────────────
    for mid, (x_m, y_m) in POSICIONES_MARCADORES.items():
        px, py = metros_a_px_ej(x_m, y_m)
        color  = (0, 220, 80)  if mid in ids_detectados else (70, 70, 70)
        radius = 10            if mid in ids_detectados else 6
        cv2.circle(mapa, (px, py), radius, color, -1)
        cv2.putText(mapa, str(mid), (px - 5, py + 5),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.45, (255, 255, 255), 1)

    # ── Paso 1: recoger posiciones de marcadores visibles ────────────
    # TODO: crea una lista con las posiciones (x_m, y_m) de los IDs detectados
    posiciones_visibles = [
        POSICIONES_MARCADORES[mid]
        for mid in ids_detectados
        if mid in POSICIONES_MARCADORES
    ]

    if not posiciones_visibles:
        cv2.putText(mapa, "Sin referencia", (PAD_ej, MAP_H_ej - 10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (100, 100, 255), 1)
        return mapa, None

    # ── Paso 2: calcular el centroide ─────────────────────────────────
    # TODO: calcula la media de X e Y de todas las posiciones visibles
    xs = [p[0] for p in posiciones_visibles]
    ys = [p[1] for p in posiciones_visibles]
    centroide_x = sum(xs) / len(xs)   # ← media de coordenadas X
    centroide_y = sum(ys) / len(ys)   # ← media de coordenadas Y

    # ── Paso 3: dibujar el dron en el centroide ───────────────────────
    px_d, py_d = metros_a_px_ej(centroide_x, centroide_y)

    # TODO: dibuja el dron como un círculo CIAN (255,220,0) de radio 14
    cv2.circle(mapa, (px_d, py_d), 14, (255, 220, 0), -1)
    cv2.circle(mapa, (px_d, py_d), 14, (255, 255, 255), 1)
    cv2.putText(mapa, "DRON", (px_d - 18, py_d - 18),
                cv2.FONT_HERSHEY_SIMPLEX, 0.45, (255, 220, 0), 1)
    cv2.putText(mapa, f"({centroide_x:.1f},{centroide_y:.1f})m",
                (px_d - 25, py_d + 28),
                cv2.FONT_HERSHEY_SIMPLEX, 0.38, (200, 200, 200), 1)

    # Líneas desde dron a cada marcador visible
    for mid in ids_detectados:
        if mid in POSICIONES_MARCADORES:
            pm = metros_a_px_ej(*POSICIONES_MARCADORES[mid])
            cv2.line(mapa, (px_d, py_d), pm, (100, 200, 255), 1)

    return mapa, (centroide_x, centroide_y)


# ── Probar con distintas combinaciones ─────────────────────────────────────
escenarios = [
    ([0],         "Solo ID 0"),
    ([0, 2],      "IDs 0 y 2 (diagonales)"),
    ([1, 3],      "IDs 1 y 3 (diagonales)"),
    ([0, 1, 2, 3], "Los 4 esquineros"),
    ([0,1,2,3,4],  "Todos los marcadores"),
]

fig, axes = plt.subplots(1, 5, figsize=(18, 4))
for ax, (ids_esc, desc) in zip(axes, escenarios):
    mapa_res, centroide = pipeline_completo(ids_esc)
    ax.imshow(cv2.cvtColor(mapa_res, cv2.COLOR_BGR2RGB))
    titulo = f"{desc}\n→ {centroide}" if centroide else desc
    ax.set_title(titulo, fontsize=8)
    ax.axis("off")

fig.suptitle("Ejercicio 5 — Localización por centroide de marcadores visibles",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

# ─── NO MODIFICAR: celda de verificación ──────────────────
print("\n🔍 VERIFICACIÓN EJERCICIO 5:")
_, c_test = pipeline_completo([0, 2])   # (0,0) y (5,4) → centroide (2.5, 2.0)
esperado  = (2.5, 2.0)
if c_test and abs(c_test[0] - esperado[0]) < 0.01 and abs(c_test[1] - esperado[1]) < 0.01:
    print(f"  ✅ Centroide correcto para IDs [0,2]: {c_test}")
else:
    print(f"  ❌ Centroide incorrecto. Esperado {esperado}, obtenido {c_test}")

_, c_test4 = pipeline_completo([0, 1, 2, 3])  # media de esquinas → centro (2.5, 2.0)
if c_test4 and abs(c_test4[0] - 2.5) < 0.01 and abs(c_test4[1] - 2.0) < 0.01:
    print(f"  ✅ Centroide correcto para los 4 esquineros: {c_test4}")
    print("\n  🎉 ¡Reto final superado! Ahora entiendes todo el pipeline.")
else:
    print(f"  ❌ Centroide de las 4 esquinas debería ser (2.5, 2.0), obtenido {c_test4}")


---

## 📚 Resumen y Conclusiones

¡Felicidades por completar la guía! A lo largo de este cuaderno has aprendido:

| Sección | Concepto clave | Herramienta/Función |
|---------|---------------|---------------------|
| 1 | Importación de librerías | `import cv2, numpy, matplotlib` |
| 2 | Anatomía y diccionarios ArUco | `cv2.aruco.getPredefinedDictionary()` |
| 3 | Modelo de cámara pinhole | Matriz intrínseca **K**, proyección 2D↔3D |
| 4 | Pipeline de detección | `detector.detectMarkers()` |
| 5 | Estimación de pose 6DOF | `cv2.solvePnP()` + ángulos de Euler |
| 6 | Mapa del almacén | Coordenadas métricas → píxeles |
| 7 | Flujo completo del detector | `procesar_frame()` |
| 8 | Límites del sistema | Ruido, distancia, número de marcadores |
| 9 | Ejercicios prácticos | Pipeline de centroide completo |

### 🚀 Próximos pasos

1. **Ejecuta `detector_simple.py`** con una cámara real apuntando a las SVG impresas.
2. **Añade solvePnP** al detector simple para obtener la altura real del dron.
3. **Explora el sistema completo** en `main.py` que incluye filtro de Kalman y mapa 3D.
4. **Calibra tu cámara** con `python -m src.camera_calibration --source 0`.

> 🧠 **Reflexión final:** Con tan solo 5 marcadores impresos, un dron puede estimar su posición  
> en tiempo real sin GPS, únicamente con una cámara y geometría. Eso es el poder de la visión artificial.

---
*Guía creada para uso docente universitario — OpenCV 4.x · NumPy · Python 3.x*
